
# Lakeflow SDP Auto CDC: Change Data Capture Made Easy

Welcome to the **Auto CDC** demo! This notebook teaches you how to automatically track and process data changes using Auto CDC in **Lakeflow Spark Declarative Pipelines (SDP)**.

---

## What you'll learn:

* **Part 1:** Understanding CDC concepts
* **Part 2:** How Auto CDC works
* **Part 3:** Basic Auto CDC setup
* **Part 4:** Advanced CDC features (SCD Type 1 & Type 2)
* **Part 5:** Handling deletes and complex scenarios
* **Part 6:** Best practices and monitoring

---

## Prerequisites:

* Complete [Lakeflow Pipeline Fundamentals](#notebook/2846436383063456)
* Complete [Lakeflow Expectations](#notebook/2846436383063443)
* Understanding of streaming tables
* Basic SQL knowledge

---

**Let's get started!**


## Part 1: Understanding Change Data Capture (CDC)

Before using Auto CDC, let's understand what CDC is and why it matters.

---


### What is Change Data Capture?

**Definition:**
* CDC tracks **changes** to data over time
* Captures INSERT, UPDATE, and DELETE operations
* Maintains history of how data evolves
* Essential for data warehousing and analytics

---

### **The Problem Without CDC:**

Imagine a customer table that gets updated daily:

**Day 1:**
```
customer_id | name      | email              | status
1           | Alice     | alice@email.com    | active
2           | Bob       | bob@email.com      | active
```

**Day 2 (Bob's email changed):**
```
customer_id | name      | email              | status
1           | Alice     | alice@email.com    | active
2           | Bob       | bob_new@email.com  | active
```

**Problems:**
* Lost Bob's old email address
* Don't know WHEN it changed
* Can't track history
* Can't audit changes

---

### **The Solution With CDC:**

CDC captures every change:

```
customer_id | name  | email              | status  | operation | timestamp
1           | Alice | alice@email.com    | active  | INSERT    | 2026-01-20
2           | Bob   | bob@email.com      | active  | INSERT    | 2026-01-20
2           | Bob   | bob_new@email.com  | active  | UPDATE    | 2026-01-21
```

**Benefits:**
* Complete history preserved
* Know what changed and when
* Can replay changes
* Full audit trail


### CDC Operation Types

CDC tracks three types of operations:

---

### **1. INSERT - New Records**

**What it means:**
* A new record was added
* First time seeing this key

**Example:**
```
customer_id | name    | operation | timestamp
3           | Charlie | INSERT    | 2026-01-22
```

---

### **2. UPDATE - Modified Records**

**What it means:**
* An existing record was changed
* Key exists, but values changed

**Example:**
```
customer_id | name  | email              | operation | timestamp
2           | Bob   | bob_new@email.com  | UPDATE    | 2026-01-21
```

---

### **3. DELETE - Removed Records**

**What it means:**
* A record was deleted from source
* Mark as deleted (soft delete) or remove (hard delete)

**Example:**
```
customer_id | operation | timestamp
1           | DELETE    | 2026-01-23
```

---

### **How CDC Identifies Operations:**

**Requires:**
1. **Primary Key** - Unique identifier (e.g., `customer_id`)
2. **Sequence Column** - Order of changes (e.g., `timestamp`, `version`)
3. **Operation Column** (optional) - Explicit operation type


### Slowly Changing Dimensions (SCD)

CDC is often used to implement **Slowly Changing Dimensions** - a data warehousing pattern.

---

### **SCD Type 1 - Overwrite (No History)**

**Behavior:**
* Overwrites old values with new values
* No history maintained
* Only current state exists

**Use case:** When history doesn't matter (e.g., fixing typos)

**Example:**
```
Before UPDATE:
customer_id | name  | email
2           | Bob   | bob@email.com

After UPDATE:
customer_id | name  | email
2           | Bob   | bob_new@email.com  ← Old email gone
```

---

### **SCD Type 2 - Track History (Full History)**

**Behavior:**
* Keeps all historical versions
* Adds new row for each change
* Tracks validity periods

**Use case:** When you need complete audit trail

**Example:**
```
customer_id | email              | valid_from  | valid_to    | is_current
2           | bob@email.com      | 2026-01-20  | 2026-01-21  | false
2           | bob_new@email.com  | 2026-01-21  | NULL        | true
```

---

**Lakeflow Auto CDC supports both SCD Type 1 and Type 2!**


## Part 2: How Lakeflow Auto CDC Works

Lakeflow Spark Declarative Pipelines (SDP) provides **automatic CDC processing** with the `AUTO CDC` flow.

---


### The Auto CDC API

**Important:** This demo uses the **Auto CDC API** from `pyspark.pipelines`, the standard API for Lakeflow Spark Declarative Pipelines.

---

### **About the API:**

The Auto CDC API replaces the deprecated `APPLY CHANGES` / `dlt.apply_changes()` APIs. The syntax is the same, but Databricks recommends using the `AUTO CDC` name going forward.

**Deprecated API:**
```python
import dlt

dlt.apply_changes(
    target="customers",
    source="customer_changes",
    keys=["customer_id"],
    sequence_by="timestamp"
)
```

**Current API:**
```python
from pyspark import pipelines as dp
from pyspark.sql.functions import col

dp.create_streaming_table("customers")

dp.create_auto_cdc_flow(
    target="customers",
    source="customer_changes",
    keys=["customer_id"],
    sequence_by=col("timestamp")
)
```

---

### **Key Points:**

1. **Import statement:** `from pyspark import pipelines as dp`
2. **Decorators:** `@dp.temporary_view`, `@dp.materialized_view`, `@dp.table` instead of `@dlt.table`
3. **Target creation:** Explicitly create the target with `dp.create_streaming_table()`
4. **CDC function:** `dp.create_auto_cdc_flow()` replaces `dlt.apply_changes()`
5. **Sequence column:** Use `col("column_name")` instead of a plain string
6. **Delete condition:** Use `expr("condition")` instead of a plain string
7. **SQL equivalent:** `CREATE FLOW ... AS AUTO CDC INTO` replaces `APPLY CHANGES INTO`

---

**Note:** The `pyspark.pipelines` module is part of Apache Spark (starting in Spark 4.1). The Auto CDC flow (`dp.create_auto_cdc_flow`) is a Databricks-specific extension not available in open-source Apache Spark Declarative Pipelines.


### What is Auto CDC?

**Auto CDC** is a Lakeflow feature that automatically:
* Processes change data from streaming sources
* Applies INSERT, UPDATE, DELETE operations
* Maintains SCD Type 1 or Type 2 tables
* Handles out-of-order data
* Deduplicates changes

---

### **Traditional CDC (Manual):**

```python
# Complex manual logic needed:
# 1. Read changes
# 2. Identify operation type
# 3. Handle duplicates
# 4. Merge with target table
# 5. Track history (if SCD Type 2)
# 6. Handle deletes
# ... 100+ lines of code
```

---

### **Auto CDC (Declarative):**

```python
from pyspark import pipelines as dp
from pyspark.sql.functions import col

# Create target table
dp.create_streaming_table("customers")

# Create Auto CDC flow
dp.create_auto_cdc_flow(
    target="customers",
    source="customer_changes",
    keys=["customer_id"],
    sequence_by=col("timestamp")
)
```

**That's it!** Lakeflow handles all the complexity.

---

### **Key Benefits:**

* **Simple** - Declarative syntax
* **Automatic** - Handles merges, deduplication
* **Reliable** - Handles out-of-order data
* **Flexible** - SCD Type 1 or Type 2
* **Performant** - Optimized for streaming


### Auto CDC Components

Auto CDC requires these components:

---

### **1. Source View (Changes)**

**What it is:**
* View with change data (streaming or batch)
* Contains INSERT, UPDATE, DELETE operations

**Example:**
```python
@dp.temporary_view
def customer_changes():
    return spark.readStream.table("bronze_customer_changes")
```

---

### **2. Target Streaming Table (Current State)**

**What it is:**
* The table to apply changes to
* Maintains current state (SCD Type 1) or history (SCD Type 2)
* Must be created BEFORE the Auto CDC flow

**Example:**
```python
dp.create_streaming_table("customers")
```

---

### **3. Auto CDC Flow**

**What it is:**
* The CDC processing logic
* Defined using `dp.create_auto_cdc_flow()`

**Required parameters:**
* `target` - Target table name
* `source` - Source view name
* `keys` - Primary key columns (list)
* `sequence_by` - Column to order changes (use `col()` function)

**Example:**
```python
from pyspark.sql.functions import col

dp.create_auto_cdc_flow(
    target="customers",
    source="customer_changes",
    keys=["customer_id"],
    sequence_by=col("timestamp")
)
```


### Auto CDC vs Delta Change Data Feed (CDF)

A common question: **Does Auto CDC require Change Data Feed to be enabled on the source table?**

**No.** Auto CDC and Delta Change Data Feed are **two different things** that are often confused.

---

### **What is Delta Change Data Feed (CDF)?**

**Delta CDF** is a Delta Lake feature that records row-level changes (inserts, updates, deletes) made to a Delta table. When enabled, Delta writes additional metadata so downstream consumers can read *only the changes* since a given version.

**How to enable CDF on a Delta table:**

```sql
-- Option 1: Enable on an existing table
ALTER TABLE my_catalog.my_schema.my_table
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)

-- Option 2: Enable at table creation
CREATE TABLE my_catalog.my_schema.my_table (...)
TBLPROPERTIES (delta.enableChangeDataFeed = true)

-- Option 3: Enable by default for all new tables in a session
SET spark.databricks.delta.properties.defaults.enableChangeDataFeed = true
```

**When is CDF automatically enabled?**
* Tables created or managed by Lakeflow Declarative Pipelines (streaming tables and materialized views) have CDF **enabled automatically** - you don't need to set it manually.
* For tables you create yourself outside of pipelines, CDF is **off by default** and must be enabled explicitly.

---

### **How Are They Different?**

| Feature | Delta Change Data Feed (CDF) | Lakeflow Auto CDC |
| --- | --- | --- |
| **What it does** | *Captures* changes made to a Delta table | *Processes* a feed of change events into a target table |
| **Direction** | Records changes on the **source** | Applies changes to the **target** |
| **Requires Delta?** | Yes — source must be a Delta table | Source can be anything (Delta, Kafka, Auto Loader, files) |
| **Requires enabling?** | Yes — must set `delta.enableChangeDataFeed = true` | No table property needed — just define the flow |
| **Output** | A change feed you can query with `table_changes()` | A fully maintained SCD Type 1 or Type 2 target table |
| **Use case** | Reading incremental changes from a Delta table | Ingesting CDC events from any upstream system |

---

### **When Would You Use Both Together?**

You might combine them in a multi-hop architecture:

1. **External CDC tool** (Debezium, AWS DMS, Fivetran) captures changes from a source database and writes them to a **Bronze Delta table**
2. **Auto CDC** processes those change events from Bronze into a **Silver streaming table** (SCD Type 1 or 2)
3. **Delta CDF** is enabled on the Silver table so that downstream Gold-layer consumers can read only incremental changes

```
Source DB → Debezium → Bronze (raw events) → Auto CDC → Silver (clean table with CDF enabled) → Gold
```

---

### **Key Takeaway:**

* **Auto CDC** does NOT require Delta CDF on the source. It works with *any* streaming source that provides change events.
* **Delta CDF** is useful when you want *downstream* consumers to efficiently read changes from a Delta table.
* **Target tables** created by `dp.create_streaming_table()` inside a pipeline get CDF enabled automatically.
* They solve different problems and complement each other in a lakehouse architecture.


## Part 3: Basic Auto CDC Setup (SCD Type 1)

Let's build our first Auto CDC pipeline with **SCD Type 1** (overwrite, no history).

**Scenario:** Track customer information with latest values only.

---

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *

# ============================================
# BRONZE LAYER: Ingest CDC Changes
# ============================================

@dp.temporary_view
def bronze_customer_changes():
    """
    Ingest customer changes from source.
    Each row represents a change event (INSERT, UPDATE, DELETE).
    """
    return (
        spark.readStream
        .table("samples.tpch.customer")
        .select(
            col("c_custkey").alias("customer_id"),
            col("c_name").alias("customer_name"),
            col("c_address").alias("address"),
            col("c_phone").alias("phone"),
            col("c_mktsegment").alias("market_segment"),
            current_timestamp().alias("change_timestamp"),
            lit("INSERT").alias("operation")  # Simulating CDC operation
        )
    )

In [0]:
# ============================================
# SILVER LAYER: Apply CDC Changes (SCD Type 1)
# ============================================

# Create the target streaming table
dp.create_streaming_table("silver_customers")

# Create Auto CDC flow to apply changes (SCD Type 1)
dp.create_auto_cdc_flow(
    target="silver_customers",
    source="bronze_customer_changes",
    keys=["customer_id"],
    sequence_by=col("change_timestamp"),
    stored_as_scd_type="1"  # SCD Type 1: Overwrite (no history)
)


### Understanding the Auto CDC Code

**What we did:**

---

### **1. Created Bronze Source View:**
```python
@dp.temporary_view
def bronze_customer_changes():
    return spark.readStream.table("samples.tpch.customer")
```
* Ingests raw change data as a view
* Each row is a change event
* Includes operation type (INSERT, UPDATE, DELETE)
* Uses `@dp.temporary_view` decorator from the `pyspark.pipelines` API

---

### **2. Created Target Streaming Table:**
```python
dp.create_streaming_table("silver_customers")
```
* Creates the target table for CDC changes
* Must be created BEFORE the Auto CDC flow
* Will contain the current state of data

---

### **3. Created Auto CDC Flow:**
```python
dp.create_auto_cdc_flow(
    target="silver_customers",           # Target table to update
    source="bronze_customer_changes",    # Source view with changes
    keys=["customer_id"],                # Primary key
    sequence_by=col("change_timestamp"), # Order changes by timestamp
    stored_as_scd_type="1"               # SCD Type 1 (overwrite)
)
```

---

### **Key Parameters Explained:**

**`target`:**
* Name of the target streaming table
* Must be created first with `create_streaming_table()`
* Contains current state of data

**`source`:**
* Name of the source view (created with `@dp.temporary_view`)
* Contains change events
* Must be a streaming source

**`keys`:**
* Primary key column(s) - list of column names
* Used to identify which record to update
* Can be composite key: `["customer_id", "order_id"]`

**`sequence_by`:**
* Column to order changes (use `col()` function)
* Ensures changes applied in correct order
* Handles out-of-order data automatically

**`stored_as_scd_type="1"`:**
* SCD Type 1: Overwrites old values
* No history maintained
* Only current state exists
* Use `"2"` for SCD Type 2 (history tracking)


### How SCD Type 1 Processes Changes

**Example scenario:**

---

### **Initial State (Empty Table):**
```
silver_customers: (empty)
```

---

### **Change 1 - INSERT:**
```
customer_id | name    | email           | operation | timestamp
1           | Alice   | alice@email.com | INSERT    | 10:00
```

**Result:**
```
silver_customers:
customer_id | name    | email
1           | Alice   | alice@email.com
```

---

### **Change 2 - INSERT:**
```
customer_id | name  | email         | operation | timestamp
2           | Bob   | bob@email.com | INSERT    | 10:01
```

**Result:**
```
silver_customers:
customer_id | name    | email
1           | Alice   | alice@email.com
2           | Bob     | bob@email.com
```

---

### **Change 3 - UPDATE:**
```
customer_id | name  | email             | operation | timestamp
2           | Bob   | bob_new@email.com | UPDATE    | 10:02
```

**Result (SCD Type 1 - Overwrite):**
```
silver_customers:
customer_id | name    | email
1           | Alice   | alice@email.com
2           | Bob     | bob_new@email.com  ← Updated (old value gone)
```

**Note:** Old email `bob@email.com` is lost - no history maintained.


### Challenge 1: Create Auto CDC for Products

**Your task:**

Create an Auto CDC pipeline for product data using SCD Type 1.

---

**Requirements:**

1. **Bronze table:** `bronze_product_changes`
   * Ingest from `samples.tpch.part`
   * Map columns:
     * `p_partkey` → `product_id`
     * `p_name` → `product_name`
     * `p_brand` → `brand`
     * `p_size` → `size`
     * `p_retailprice` → `price`
   * Add `change_timestamp` (current timestamp)
   * Add `operation` (set to "INSERT")

2. **Streaming source:** `product_changes_stream`
   * Read from `bronze_product_changes`

3. **Apply CDC:**
   * Target: `silver_products`
   * Keys: `["product_id"]`
   * Sequence by: `change_timestamp`
   * SCD Type 1

---

**Write your code below:**

In [0]:
# ============================================
# CHALLENGE 1: Your solution here
# ============================================

# TODO: Create bronze_product_changes table


# TODO: Create product_changes_stream table


# TODO: Apply CDC changes to silver_products



## Part 4: Advanced CDC - SCD Type 2 (History Tracking)

Now let's implement **SCD Type 2** to maintain complete history of changes.

**Scenario:** Track customer information with full audit trail.

---


### SCD Type 2 - History Tracking

**What is SCD Type 2?**
* Maintains **complete history** of all changes
* Creates **new row** for each change
* Tracks **validity periods** for each version
* Marks **current** vs **historical** records

---

### **SCD Type 2 Columns:**

Auto CDC automatically adds these columns:

**`__START_AT`:**
* When this version became valid
* Timestamp of the change

**`__END_AT`:**
* When this version became invalid
* `NULL` for the current (active) version

**How to identify current records:**
* Current records: `__END_AT IS NULL`
* Historical records: `__END_AT IS NOT NULL`

---

### **Example:**

**Changes:**
```
Timestamp | customer_id | name  | email
10:00     | 2           | Bob   | bob@email.com
10:05     | 2           | Bob   | bob_new@email.com
10:10     | 2           | Bob   | bob_final@email.com
```

**Result (SCD Type 2):**
```
customer_id | name | email              | __START_AT | __END_AT
2           | Bob  | bob@email.com      | 10:00      | 10:05
2           | Bob  | bob_new@email.com  | 10:05      | 10:10
2           | Bob  | bob_final@email.com| 10:10      | NULL      ← current record
```

**Benefits:**
* Complete audit trail
* Can query historical state
* Know exactly when changes occurred
* Can rollback to any point in time
* Current record identified by `__END_AT IS NULL`

In [0]:
# ============================================
# SILVER LAYER: Apply CDC Changes (SCD Type 2)
# ============================================

# Create the target streaming table for SCD Type 2
dp.create_streaming_table("silver_customers_history")

# Create Auto CDC flow with SCD Type 2 (history tracking)
dp.create_auto_cdc_flow(
    target="silver_customers_history",
    source="bronze_customer_changes",
    keys=["customer_id"],
    sequence_by=col("change_timestamp"),
    stored_as_scd_type="2"  # SCD Type 2: Track history
)


### SCD Type 2 - What Changed?

**The only difference from SCD Type 1:**

```python
stored_as_scd_type="2"  # Changed from "1" to "2"
```

**That's it!** Auto CDC handles all the complexity:

---

### **What Auto CDC Does Automatically:**

1. **Creates history rows:**
   * New row for each change
   * Preserves all historical versions

2. **Adds tracking columns:**
   * `__START_AT` - When version became valid
   * `__END_AT` - When version became invalid (`NULL` for the current record)

3. **Manages validity periods:**
   * Sets `__END_AT` on old versions
   * Sets `__END_AT = NULL` on current version

4. **Handles out-of-order data:**
   * Uses `sequence_by` to order changes
   * Correctly updates validity periods

---

### **Querying SCD Type 2 Tables:**

**Get current records only:**
```sql
SELECT * FROM silver_customers_history
WHERE __END_AT IS NULL
```

**Get all history:**
```sql
SELECT * FROM silver_customers_history
ORDER BY customer_id, __START_AT
```

**Get state at specific time:**
```sql
SELECT * FROM silver_customers_history
WHERE __START_AT <= '2026-01-25 10:00:00'
  AND (__END_AT > '2026-01-25 10:00:00' OR __END_AT IS NULL)
```


### Challenge 2: Create SCD Type 2 for Suppliers

**Your task:**

Create an Auto CDC pipeline for supplier data with full history tracking (SCD Type 2).

---

**Requirements:**

1. **Bronze table:** `bronze_supplier_changes`
   * Ingest from `samples.tpch.supplier`
   * Map columns:
     * `s_suppkey` → `supplier_id`
     * `s_name` → `supplier_name`
     * `s_address` → `address`
     * `s_phone` → `phone`
     * `s_acctbal` → `account_balance`
   * Add `change_timestamp` (current timestamp)
   * Add `operation` (set to "INSERT")

2. **Streaming source:** `supplier_changes_stream`
   * Read from `bronze_supplier_changes`

3. **Apply CDC with SCD Type 2:**
   * Target: `silver_suppliers_history`
   * Keys: `["supplier_id"]`
   * Sequence by: `change_timestamp`
   * SCD Type 2 (history tracking)

---

**Write your code below:**

In [0]:
# ============================================
# CHALLENGE 2: Your solution here
# ============================================

# TODO: Create bronze_supplier_changes table


# TODO: Create supplier_changes_stream table


# TODO: Apply CDC changes with SCD Type 2



## Part 5: Handling Deletes and Complex Scenarios

Let's explore advanced CDC features: handling deletes, column tracking, and filtering.

---


### Handling DELETE Operations

Auto CDC can handle DELETE operations in two ways:

---

### **1. Soft Delete (Default for SCD Type 2)**

**Behavior:**
* Marks record as deleted
* Keeps the record in the table
* Sets `__END_AT` timestamp
* Record becomes historical (`__END_AT IS NOT NULL`)

**Use case:** Maintain complete audit trail including deletions

**Example:**
```
Before DELETE:
customer_id | name  | __START_AT | __END_AT
2           | Bob   | 10:00      | NULL       ← current record

After DELETE:
customer_id | name  | __START_AT | __END_AT
2           | Bob   | 10:00      | 10:15      ← no longer current
```

---

### **2. Hard Delete (Optional)**

**Behavior:**
* Physically removes the record
* No trace left in table

**Use case:** Compliance (GDPR), data retention policies

**Configuration (Python):**
```python
from pyspark import pipelines as dp
from pyspark.sql.functions import col, expr

dp.create_streaming_table("customers")

dp.create_auto_cdc_flow(
    target="customers",
    source="customer_changes",
    keys=["customer_id"],
    sequence_by=col("timestamp"),
    apply_as_deletes=expr("operation = 'DELETE'"),       # Specify delete condition
    apply_as_truncates=expr("operation = 'TRUNCATE'")    # Optional: truncate condition
)
```

---

### **Identifying Deletes:**

Auto CDC needs to know which records are deletes:

**Option 1: Operation column**
```python
apply_as_deletes=expr("operation = 'DELETE'")
```

**Option 2: Deleted flag**
```python
apply_as_deletes=expr("is_deleted = true")
```

**Option 3: Null values**
```python
apply_as_deletes=expr("customer_name IS NULL")
```

In [0]:
# ============================================
# CDC with DELETE Handling
# ============================================

# Bronze layer with delete operations
@dp.temporary_view
def bronze_customer_changes_with_deletes():
    """
    Simulating CDC feed with INSERT, UPDATE, and DELETE operations.
    """
    return (
        spark.readStream
        .table("samples.tpch.customer")
        .select(
            col("c_custkey").alias("customer_id"),
            col("c_name").alias("customer_name"),
            col("c_address").alias("address"),
            col("c_phone").alias("phone"),
            current_timestamp().alias("change_timestamp"),
            # Randomly mark some records as deleted for demo
            when(col("c_custkey") % 100 == 0, lit("DELETE"))
            .otherwise(lit("INSERT"))
            .alias("operation")
        )
    )

# Create target streaming table
dp.create_streaming_table("silver_customers_with_deletes")

# Apply changes with delete handling
dp.create_auto_cdc_flow(
    target="silver_customers_with_deletes",
    source="bronze_customer_changes_with_deletes",
    keys=["customer_id"],
    sequence_by=col("change_timestamp"),
    stored_as_scd_type="2",
    apply_as_deletes=expr("operation = 'DELETE'")  # Handle DELETE operations
)


### Advanced Features: Column Tracking and Filtering

---

### **1. Track History on Specific Columns**

You can exclude columns from history tracking (SCD Type 2):

```python
dp.create_auto_cdc_flow(
    target="customers",
    source="customer_changes",
    keys=["customer_id"],
    sequence_by=col("timestamp"),
    stored_as_scd_type="2",
    track_history_except_column_list=["last_login", "login_count"]  # Don't track these
)
```

**Use case:**
* Ignore changes to non-critical columns
* Reduce storage for SCD Type 2
* Focus on important business attributes

**Example:**
* Track changes to `email`, `phone`, `address`
* Ignore changes to `last_login`, `login_count` (frequently changing, not important)

---

### **2. Exclude Columns from Target**

Exclude columns from the target table:

```python
dp.create_auto_cdc_flow(
    target="customers",
    source="customer_changes",
    keys=["customer_id"],
    sequence_by=col("timestamp"),
    except_column_list=["operation", "source_system"]  # Don't include these
)
```

**Use case:**
* Remove CDC metadata columns
* Exclude technical columns
* Keep target table clean

---

### **3. Filter Source Data**

Apply filters before CDC processing:

```python
@dp.temporary_view
def customer_changes_filtered():
    return (
        spark.readStream.table("bronze_customer_changes")
        .filter(col("country") == "USA")  # Only process USA customers
    )

dp.create_streaming_table("silver_customers_usa")

dp.create_auto_cdc_flow(
    target="silver_customers_usa",
    source="customer_changes_filtered",
    keys=["customer_id"],
    sequence_by=col("timestamp")
)
```


### Composite Keys (Multiple Columns)

Some tables require multiple columns as primary key:

---

### **Example: Order Line Items**

**Scenario:**
* Each order has multiple line items
* Primary key: `order_id` + `line_number`

**Implementation:**
```python
@dp.temporary_view
def order_line_changes():
    return (
        spark.readStream
        .table("samples.tpch.lineitem")
        .select(
            col("l_orderkey").alias("order_id"),
            col("l_linenumber").alias("line_number"),
            col("l_partkey").alias("product_id"),
            col("l_quantity").alias("quantity"),
            col("l_extendedprice").alias("price"),
            current_timestamp().alias("change_timestamp")
        )
    )

# Create target table
dp.create_streaming_table("silver_order_lines")

# Create Auto CDC flow with composite key
dp.create_auto_cdc_flow(
    target="silver_order_lines",
    source="order_line_changes",
    keys=["order_id", "line_number"],  # Composite key
    sequence_by=col("change_timestamp"),
    stored_as_scd_type="1"
)
```

**Key points:**
* `keys` accepts list of multiple columns
* All key columns must be present in source
* Combination must be unique


### Challenge 3: Advanced CDC with Composite Keys

**Your task:**

Create an Auto CDC pipeline for order line items with composite keys and delete handling.

---

**Requirements:**

1. **Bronze table:** `bronze_orderline_changes`
   * Ingest from `samples.tpch.lineitem`
   * Map columns:
     * `l_orderkey` → `order_id`
     * `l_linenumber` → `line_number`
     * `l_partkey` → `product_id`
     * `l_quantity` → `quantity`
     * `l_extendedprice` → `extended_price`
     * `l_linestatus` → `status`
   * Add `change_timestamp` (current timestamp)
   * Add `operation` column:
     * Set to "DELETE" if `l_linestatus = 'F'` (finished)
     * Otherwise set to "INSERT"

2. **Streaming source:** `orderline_changes_stream`
   * Read from `bronze_orderline_changes`

3. **Apply CDC:**
   * Target: `silver_order_lines`
   * Composite keys: `["order_id", "line_number"]`
   * Sequence by: `change_timestamp`
   * SCD Type 1
   * Handle deletes: `operation = 'DELETE'`
   * Exclude columns: `["operation"]`

---

**Write your code below:** 👇

In [0]:
# ============================================
# CHALLENGE 3: Your solution here
# ============================================

# TODO: Create bronze_orderline_changes table


# TODO: Create orderline_changes_stream table


# TODO: Apply CDC with composite keys and delete handling


## Congratulations!

You've completed the Lakeflow SDP Auto CDC demo!

### **What You Learned:**

* **CDC Basics** - Tracks INSERT, UPDATE, DELETE operations for data warehousing  
* **Auto CDC API** - Import `pyspark.pipelines`, create targets with `dp.create_streaming_table()`, create flows with `dp.create_auto_cdc_flow()`  
* **SCD Type 1** - Overwrite with no history using `stored_as_scd_type="1"`  
* **SCD Type 2** - Track full history with `stored_as_scd_type="2"`, automatic `__START_AT` and `__END_AT` columns  
* **Delete handling** - Use `apply_as_deletes` to process DELETE operations  
* **Composite keys** - Use multiple columns as primary key with `keys=[...]`  
* **Column filtering** - Exclude columns with `except_column_list` and `track_history_except_column_list`  
* **SQL equivalent** - `CREATE FLOW ... AS AUTO CDC INTO` for SQL-based pipelines  

---

### **Key Takeaways:**

1. **Auto CDC handles complexity** - Automatic merge, deduplication, and out-of-order data
2. **Two SCD types for two needs** - Type 1 overwrites, Type 2 tracks history
3. **Bronze ingests, silver applies CDC** - General medallion pattern
4. **Sequence by is critical** - Ensures correct ordering of change events
5. **Composite keys for multi-column identity** - All key columns must be present in source
6. **Deletes need explicit configuration** - Use `apply_as_deletes=expr("condition")`
7. **Current SCD Type 2 records** - Identified by `__END_AT IS NULL`

---

### **Next Steps:**

* Create your pipeline in the Lakeflow Pipelines Editor
* Add this notebook as source code and configure target catalog and schema
* Run the pipeline and check event logs
* Validate CDC processing and monitor results
* Combine with expectations and data quality rules
* Optimize performance for production workloads

---

### **Resources:**

* [Lakeflow Pipeline Fundamentals](#notebook/2846436383063456)
* [Lakeflow Expectations](#notebook/2846436383063443)
* [Auto CDC API Documentation](https://docs.databricks.com/aws/en/ldp/cdc/)
* [Change Data Capture Concepts](https://docs.databricks.com/aws/en/ldp/what-is-change-data-capture/)


# Complete Solutions

**Only look at these if you're stuck or want to verify your work!**

Try to solve the challenges yourself first. Learning happens through struggle and problem-solving!

---

## Solution: Challenge 1 (Auto CDC for Products - SCD Type 1)
<br/>
<details>

```python
# ============================================
# CHALLENGE 1: Auto CDC for Products (SCD Type 1)
# ============================================

# Bronze: Ingest product changes from source
@dp.temporary_view
def bronze_product_changes():
    """
    Ingest product changes from samples.tpch.part.
    Each row represents a change event.
    """
    return (
        spark.readStream
        .table("samples.tpch.part")
        .select(
            col("p_partkey").alias("product_id"),
            col("p_name").alias("product_name"),
            col("p_brand").alias("brand"),
            col("p_size").alias("size"),
            col("p_retailprice").alias("price"),
            current_timestamp().alias("change_timestamp"),
            lit("INSERT").alias("operation")
        )
    )

# Streaming source: Read from bronze_product_changes
@dp.temporary_view
def product_changes_stream():
    """
    Streaming source reading from bronze product changes.
    """
    return spark.readStream.table("bronze_product_changes")

# Create target streaming table
dp.create_streaming_table("silver_products")

# Apply Auto CDC (SCD Type 1)
dp.create_auto_cdc_flow(
    target="silver_products",
    source="product_changes_stream",
    keys=["product_id"],
    sequence_by=col("change_timestamp"),
    stored_as_scd_type="1"
)
```

**Key concepts:**
* `@dp.temporary_view` creates the bronze source view
* A separate `product_changes_stream` view reads from the bronze view
* `dp.create_streaming_table()` creates the target before the CDC flow
* `stored_as_scd_type="1"` overwrites old values (no history)
* `keys=["product_id"]` uses the part key as primary key

</details>

## Solution: Challenge 2 (SCD Type 2 for Suppliers)
<br/>
<details>

```python
# ============================================
# CHALLENGE 2: Auto CDC for Suppliers (SCD Type 2)
# ============================================

# Bronze: Ingest supplier changes from source
@dp.temporary_view
def bronze_supplier_changes():
    """
    Ingest supplier changes from samples.tpch.supplier.
    Each row represents a change event.
    """
    return (
        spark.readStream
        .table("samples.tpch.supplier")
        .select(
            col("s_suppkey").alias("supplier_id"),
            col("s_name").alias("supplier_name"),
            col("s_address").alias("address"),
            col("s_phone").alias("phone"),
            col("s_acctbal").alias("account_balance"),
            current_timestamp().alias("change_timestamp"),
            lit("INSERT").alias("operation")
        )
    )

# Streaming source: Read from bronze_supplier_changes
@dp.temporary_view
def supplier_changes_stream():
    """
    Streaming source reading from bronze supplier changes.
    """
    return spark.readStream.table("bronze_supplier_changes")

# Create target streaming table
dp.create_streaming_table("silver_suppliers_history")

# Apply Auto CDC (SCD Type 2 - History Tracking)
dp.create_auto_cdc_flow(
    target="silver_suppliers_history",
    source="supplier_changes_stream",
    keys=["supplier_id"],
    sequence_by=col("change_timestamp"),
    stored_as_scd_type="2"
)
```

**Key concepts:**
* Only change from Challenge 1: `stored_as_scd_type="2"` enables full history tracking
* Auto CDC automatically adds `__START_AT`, `__END_AT`, and `__CURRENT` columns
* Each change creates a new row instead of overwriting
* Current records have `__END_AT = NULL` and `__CURRENT = true`
* Historical records have `__CURRENT = false` with validity periods

</details>

## Solution: Challenge 3 (Advanced CDC with Composite Keys)
<br/>
<details>

```python
# ============================================
# CHALLENGE 3: Advanced CDC with Composite Keys
# ============================================

# Bronze: Ingest order line changes with delete operations
@dp.temporary_view
def bronze_orderline_changes():
    """
    Ingest order line changes from samples.tpch.lineitem.
    Marks finished (status 'F') lines as DELETE operations.
    """
    return (
        spark.readStream
        .table("samples.tpch.lineitem")
        .select(
            col("l_orderkey").alias("order_id"),
            col("l_linenumber").alias("line_number"),
            col("l_partkey").alias("product_id"),
            col("l_quantity").alias("quantity"),
            col("l_extendedprice").alias("extended_price"),
            col("l_linestatus").alias("status"),
            current_timestamp().alias("change_timestamp"),
            when(col("l_linestatus") == "F", lit("DELETE"))
                .otherwise(lit("INSERT"))
                .alias("operation")
        )
    )

# Streaming source: Read from bronze_orderline_changes
@dp.temporary_view
def orderline_changes_stream():
    """
    Streaming source reading from bronze order line changes.
    """
    return spark.readStream.table("bronze_orderline_changes")

# Create target streaming table
dp.create_streaming_table("silver_order_lines")

# Apply Auto CDC with composite keys and delete handling
dp.create_auto_cdc_flow(
    target="silver_order_lines",
    source="orderline_changes_stream",
    keys=["order_id", "line_number"],
    sequence_by=col("change_timestamp"),
    stored_as_scd_type="1",
    apply_as_deletes=expr("operation = 'DELETE'"),
    except_column_list=["operation"]
)
```

**Key concepts:**
* **Composite keys:** `keys=["order_id", "line_number"]` uses two columns as the primary key
* **Delete handling:** `apply_as_deletes=expr("operation = 'DELETE'")` tells Auto CDC which rows are deletes
* **Conditional operation column:** Uses `when/otherwise` to set operation based on `l_linestatus`
* **Exclude columns:** `except_column_list=["operation"]` removes the CDC metadata column from the target table
* Lines with `l_linestatus = 'F'` (finished) are treated as DELETE operations
* All other lines are treated as INSERT operations

</details>